In [ ]:
from antares_client.search import search
from antares_client.search import get_by_id, get_by_ztf_object_id, get_by_lsst_dia_object_id


import requests
from io import BytesIO
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from astropy.io import fits
from astropy.visualization import make_lupton_rgb

In [ ]:
QUERY = {
    "query": {
        "bool": {
            "filter": {
                "terms": {
                    "tags": [
                        "lantern_xgboost_t2.0.7_c0.95"
                    ]
                }
            }
        }
    }
}


BASE_URL = "https://www.legacysurvey.org/viewer/cutout"

In [ ]:
def process_locus(locus):
    
    print(locus.locus_id)

    ra = locus.ra
    dec = locus.dec

    return ra, dec


def _build_url(ra, dec, fmt="jpg", layer="ls-dr10", pixscale=0.263,
               size=50, bands=None):
    params = {"ra": ra, "dec": dec, "layer": layer,
              "pixscale": pixscale, "size": size}
    if bands is not None:
        params["bands"] = bands
    qs = "&".join(f"{k}={v}" for k, v in params.items())
    return f"{BASE_URL}.{fmt}?{qs}"


def show_cutout(ra, dec, size=50, layer="ls-dr10", pixscale=0.263,
                title=None, figsize=(3, 3)):
    """Fetch a JPG cutout from Legacy Survey and display it inline."""
    url = _build_url(ra, dec, "jpg", layer, pixscale, size)
    r = requests.get(url, timeout=30); r.raise_for_status()
    img = PILImage.open(BytesIO(r.content))

    fov = size * pixscale / 60.0
    plt.figure(figsize=figsize)
    plt.imshow(img)
    plt.title(title or f"RA={ra:.4f}, Dec={dec:+.4f}\n({fov:.2f}′|{size}pix, {layer})")
    plt.xticks([]); plt.yticks([])
    plt.show()
    return url


def show_fits_rgb(ra, dec, size=50, layer="ls-dr10", pixscale=0.263,
                  bands="grz", stretch=0.05, Q=5, minimum=-0.05,
                  title=None, figsize=(3, 3), return_data=False):
    """
    Fetch a grz FITS cutout and render it as a Lupton RGB image.
    Channel mapping: z → R, r → G, g → B.
    """
    url = _build_url(ra, dec, "fits", layer, pixscale, size, bands=bands)
    r = requests.get(url, timeout=60); r.raise_for_status()
    hdul = fits.open(BytesIO(r.content))

    cube = hdul[0].data
    if cube.ndim != 3 or cube.shape[0] != len(bands):
        raise ValueError(f"Unexpected FITS shape {cube.shape} for bands={bands}")

    imgs = {b: cube[i] for i, b in enumerate(bands)}
    rgb = make_lupton_rgb(imgs["z"], imgs["r"], imgs["g"],
                          minimum=minimum, stretch=stretch, Q=Q)

    fov = size * pixscale / 60.0
    plt.figure(figsize=figsize)
    plt.imshow(rgb, origin="lower")
    plt.title(title or f"RA={ra:.4f}, Dec={dec:+.4f}\n"
                       f"({fov:.2f}′|{size}pix, {layer}, Lupton g/r/z)")
    plt.xticks([]); plt.yticks([])
    plt.show()

    #return (rgb, hdul, url) if return_data else url

In [ ]:
#ra, dec = 28.9715, 1.2707
#show_cutout(ra, dec, size=50)
#show_fits_rgb(ra, dec, size=50)

In [ ]:
def scan_lupton(ra, dec, size=256, layer="ls-dr10", pixscale=0.262,
                stretches=(0.5, 0.1, 0.05, 0.01),
                Qs=(5, 10, 20), minimum=-0.05):
    """Grid of Lupton RGB renderings over (stretch, Q)."""
    url = _build_url(ra, dec, "fits", layer, pixscale, size, bands="grz")
    r = requests.get(url, timeout=60); r.raise_for_status()
    cube = fits.open(BytesIO(r.content))[0].data
    g, r_, z = cube[0], cube[1], cube[2]

    nrows, ncols = len(Qs), len(stretches)
    fig, axes = plt.subplots(nrows, ncols, figsize=(3 * ncols, 3 * nrows))
    axes = axes.reshape(nrows, ncols)
    for i, Q in enumerate(Qs):
        for j, s in enumerate(stretches):
            rgb = make_lupton_rgb(z, r_, g, minimum=minimum, stretch=s, Q=Q)
            ax = axes[i, j]
            ax.imshow(rgb, origin="lower")
            ax.set_title(f"stretch={s}, Q={Q}", fontsize=9)
            ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout(); plt.show()

In [ ]:
#scan_lupton(ra, dec, size=50)

In [ ]:
def show_by_id(locus_id):
    locus = get_by_id(locus_id)

    ra, dec = process_locus(locus)
    show_cutout(ra, dec, size=50)
    show_fits_rgb(ra, dec, size=50)

In [ ]:
# Some known lenses
# DES
locus_id = "ANT20268nau1wefp9jb"
show_by_id(locus_id)

In [ ]:
for ind, locus in enumerate(search(QUERY)):
    #if ind==50: break
    print(ind)
    ra, dec = process_locus(locus)
    show_cutout(ra, dec, size=50)
    #show_fits_rgb(ra, dec, size=50)